# Fire Detection — YOLOv8 Training on Kaggle

This notebook trains a small YOLOv8 model to detect fire in images. When it finishes, you get a file called `best.pt` that you can download and run on your laptop.

## Before you click Run All

Two things on the right-hand sidebar:

1. **Settings → Accelerator → GPU T4 x2.** Without this, training will be ~30x slower. Kaggle makes you verify your phone number the first time you turn on a GPU; it's free.
2. **Add Input → attach a fire-detection dataset.** Any Kaggle dataset that contains a `data.yaml` file in YOLO format will work. Search Kaggle for "fire detection yolo" and pick one, or upload your own.

Expected runtime: about 30–45 minutes on a T4 with ~2000 images and 80 epochs.

## Step 1 — Check the GPU is on

If this prints `False`, go to Settings on the right and switch the Accelerator to GPU.

In [ ]:
import torch

gpu_ok = torch.cuda.is_available()
print(f"CUDA available: {gpu_ok}")
if gpu_ok:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    raise RuntimeError(
        "No GPU detected. Open the Settings panel on the right and set\n"
        "Accelerator to 'GPU T4 x2', then re-run this cell."
    )

## Step 2 — Install / upgrade ultralytics

Kaggle pre-installs ultralytics but the version is often old. This makes sure we have a current one.

In [ ]:
!pip install -q -U ultralytics

import ultralytics
print(f"ultralytics version: {ultralytics.__version__}")

## Step 3 — Find the dataset (robust)

We auto-detect the attached dataset by looking for a `data.yaml` file anywhere under `/kaggle/input/`. Then — instead of trusting the paths inside that YAML (they're often wrong: they point to `val/` but the folder is `valid/`, or they're missing a sub-directory level) — we scan the filesystem ourselves to find the real `train/` and `val`/`valid/` image folders, and write a fresh `data.yaml` to `/kaggle/working/`.

This means basically any YOLO-format dataset on Kaggle will Just Work, even if its yaml is broken.

In [ ]:
import os
import yaml
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT  = Path('/kaggle/working')

# Folder name patterns we skip because they're NEVER the real dataset
SKIP_PARTS = {'runs', 'predict', 'predict_sample', 'weights', '.ipynb_checkpoints', '__pycache__'}

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def is_skipped(p: Path) -> bool:
    return any(part in SKIP_PARTS for part in p.parts)

def count_images(p: Path) -> int:
    try:
        return sum(1 for c in p.iterdir() if c.is_file() and c.suffix.lower() in IMG_EXTS)
    except Exception:
        return 0

# 1) Find candidate data.yaml files (skip ones inside output folders like runs/)
yaml_candidates = [
    y for y in INPUT_ROOT.rglob('data.yaml')
    if not is_skipped(y) and 'notebooks' not in y.parts
]

# If everything got filtered out (e.g. user attached a notebook by mistake),
# fall back to any data.yaml so we can still report something useful
if not yaml_candidates:
    yaml_candidates = list(INPUT_ROOT.rglob('data.yaml'))

if not yaml_candidates:
    raise FileNotFoundError(
        "No data.yaml found under /kaggle/input.\n"
        "Attach a YOLO-format fire dataset using 'Add Input' on the right.\n"
        "Make sure you're on the 'Datasets' tab, not 'Notebooks'."
    )
src_yaml = yaml_candidates[0]
with open(src_yaml) as f:
    original_cfg = yaml.safe_load(f) or {}
print(f"Found dataset config : {src_yaml}")

# Detect "you attached a notebook by mistake" early
if 'notebooks' in src_yaml.parts:
    print("\n" + "!" * 60)
    print("WARNING: this input is a NOTEBOOK, not a dataset.")
    print("Detach it and attach a real dataset from the 'Datasets' tab.")
    print("!" * 60 + "\n")

# 2) Scan for actual image folders, skipping known-junk locations
dataset_root = src_yaml.parent

image_dirs = []
for p in dataset_root.rglob('images'):
    if p.is_dir() and not is_skipped(p) and count_images(p) > 0:
        image_dirs.append(p)

# Fallback: some datasets put images directly in train/ instead of train/images/
if not image_dirs:
    for split_name in ('train', 'val', 'valid', 'test'):
        for p in dataset_root.rglob(split_name):
            if p.is_dir() and not is_skipped(p) and count_images(p) > 0:
                image_dirs.append(p)

if not image_dirs:
    tree_preview = "\n".join(
        str(p.relative_to(dataset_root))
        for p in dataset_root.rglob('*') if p.is_dir()
    )[:2000]
    raise FileNotFoundError(
        f"Couldn't find any real image folders under {dataset_root}.\n"
        f"Folder tree:\n{tree_preview}"
    )

# 3) Classify image dirs as train / val / test by their path
splits = {'train': None, 'val': None, 'test': None}
for d in image_dirs:
    parts_lc = [s.lower() for s in d.parts]
    if any('train' in s for s in parts_lc) and splits['train'] is None:
        splits['train'] = d
    elif any(('val' in s or 'valid' in s) for s in parts_lc) and splits['val'] is None:
        splits['val'] = d
    elif any('test' in s for s in parts_lc) and splits['test'] is None:
        splits['test'] = d

# If no val, use test as val. If no test either, use train (loud warning).
if splits['val'] is None and splits['test'] is not None:
    splits['val'] = splits['test']
    print("Note: no 'val' folder, using 'test' folder for validation.")

if splits['train'] is None:
    # last-resort: pick the largest image folder as train
    splits['train'] = max(image_dirs, key=count_images)
    print(f"WARNING: no 'train' folder found, using largest folder: {splits['train']}")

if splits['val'] is None:
    splits['val'] = splits['train']
    print("WARNING: no 'val'/'valid'/'test' folder found, using train as val (metrics will be optimistic).")

# 4) Class names from original yaml, fall back to ['fire']
names = original_cfg.get('names') or ['fire']
if isinstance(names, dict):
    names = [names[k] for k in sorted(names)]

cfg = {
    'path' : str(dataset_root),
    'train': str(splits['train']),
    'val'  : str(splits['val']),
    'nc'   : len(names),
    'names': names,
}
if splits['test'] is not None and splits['test'] != splits['val']:
    cfg['test'] = str(splits['test'])

dst_yaml = WORK_ROOT / 'data.yaml'
with open(dst_yaml, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

# 5) Summary + sanity check
n_train = count_images(Path(cfg['train']))
n_val   = count_images(Path(cfg['val']))

print()
print("Final dataset config")
print("--------------------")
print(f"Classes      : {cfg['names']}")
print(f"Train images : {cfg['train']}")
print(f"Val images   : {cfg['val']}")
if 'test' in cfg:
    print(f"Test images  : {cfg['test']}")
print(f"Counts       : {n_train} train / {n_val} val")
print(f"Written      : {dst_yaml}")

# Sanity check: refuse to continue with absurdly small datasets
if n_train < 100:
    raise RuntimeError(
        f"\n\nOnly {n_train} training images found. That's far too few — "
        f"you almost certainly attached the wrong thing.\n"
        f"Detach the current input, then 'Add Input' from the DATASETS tab\n"
        f"(not Notebooks) and search for 'd-fire' or 'fire detection yolo'."
    )

### (Optional) Use Roboflow instead

If you'd rather pull the dataset from Roboflow instead of using a Kaggle Dataset, uncomment the cell below and paste your API key. You usually don't need this — attached Kaggle Datasets are faster and don't need an API key.

In [ ]:
# !pip install -q roboflow
# from roboflow import Roboflow
# Configure any dataset-provider credentials through the provider's secure interface; do not store them in this notebook.
# project = rf.workspace("your-workspace").project("your-project")
# dataset = project.version(1).download("yolov8")
# dst_yaml = f"{dataset.location}/data.yaml"

## Step 4 — Train

### Pick a model size

You're running this on a laptop, so you can afford something bigger than the nano model. Change `MODEL_SIZE` below:

| Size | File         | Speed on laptop CPU | Speed on laptop GPU | Accuracy |
|------|--------------|---------------------|---------------------|----------|
| `n`  | yolov8n.pt   | fast (15-30 FPS)    | very fast (100+)    | lowest   |
| `s`  | yolov8s.pt   | okay (8-15 FPS)     | fast (60-100)       | **good — recommended** |
| `m`  | yolov8m.pt   | slow (3-6 FPS)      | okay (30-60)        | better   |
| `l`  | yolov8l.pt   | painful             | okay (20-40)        | best     |

Default below is **`s`** (small). If you have no GPU on your laptop, drop to `n`. If you have a strong NVIDIA card and want max accuracy, try `m` or `l`.

Training time on a Kaggle T4 (free GPU) for ~2000 images, 80 epochs:
- `n`: ~25 min
- `s`: ~35 min
- `m`: ~60 min
- `l`: ~90 min

### Other knobs you can tweak

- **`epochs`** — how many times the model sees the whole dataset. 80 is a solid default.
- **`imgsz`** — input image size. Bigger = more accurate, slower. 640 is the YOLO standard.
- **`batch`** — bigger needs more GPU memory. 16 is safe for `n`/`s` on a T4; lower it to 8 for `m`, 4 for `l` if you hit out-of-memory errors.
- **`patience`** — stop early if validation hasn't improved in this many epochs.

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import shutil
import torch

# Change this one letter to pick model size: 'n' | 's' | 'm' | 'l'
MODEL_SIZE = 's'

# Auto-pick a safe batch size for each model on a single 16 GB GPU (T4 or P100)
BATCH_PER_GPU = {'n': 16, 's': 16, 'm': 8, 'l': 4}

# Auto-detect GPU setup
n_gpus = torch.cuda.device_count()
device = list(range(n_gpus)) if n_gpus > 1 else 0
batch  = BATCH_PER_GPU[MODEL_SIZE] * max(n_gpus, 1)

print(f"GPUs detected : {n_gpus}")
for i in range(n_gpus):
    print(f"  [{i}] {torch.cuda.get_device_name(i)}")
print(f"Device        : {device}")
print(f"Batch size    : {batch}")

# ---------------------------------------------------------------------------
# Resume support
# ---------------------------------------------------------------------------
# Look for a previously-interrupted run in three places, in priority order:
#   1) The current session's run dir (interactive: cell re-run after crash)
#   2) /kaggle/working/last.pt that we manually checkpointed before
#   3) Any previous-commit output attached as /kaggle/input/<name>/last.pt
# If none exist, start fresh from yolov8<size>.pt.

run_dir   = WORK_ROOT / 'runs' / 'fire'
last_here = run_dir / 'weights' / 'last.pt'
last_root = WORK_ROOT / 'last.pt'

# Search /kaggle/input/ for last.pt files we might have attached from a prior run
input_last_candidates = list(Path('/kaggle/input').rglob('last.pt'))

resume_from = None
if last_here.exists():
    resume_from = last_here
    print(f"\nResuming from current-session checkpoint: {resume_from}")
elif last_root.exists():
    # Restore it into the run dir so ultralytics can resume cleanly
    run_dir.joinpath('weights').mkdir(parents=True, exist_ok=True)
    shutil.copy2(last_root, last_here)
    resume_from = last_here
    print(f"\nResuming from /kaggle/working/last.pt -> {resume_from}")
elif input_last_candidates:
    src = input_last_candidates[0]
    run_dir.joinpath('weights').mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, last_here)
    # Also copy best.pt if it's next to it
    best_neighbour = src.parent / 'best.pt'
    if best_neighbour.exists():
        shutil.copy2(best_neighbour, run_dir / 'weights' / 'best.pt')
    resume_from = last_here
    print(f"\nResuming from attached input: {src}")
else:
    print(f"\nNo checkpoint found — starting fresh from yolov8{MODEL_SIZE}.pt")

# ---------------------------------------------------------------------------
# Train
# ---------------------------------------------------------------------------
if resume_from is not None:
    # When resuming, load the checkpoint and pass resume=True.
    # All other hyperparameters come from the checkpoint, so we don't repeat them.
    model = YOLO(str(resume_from))
    results = model.train(resume=True)
else:
    model = YOLO(f'yolov8{MODEL_SIZE}.pt')
    results = model.train(
        data=str(dst_yaml),
        epochs=80,
        imgsz=640,
        batch=batch,
        device=device,
        patience=15,
        project=str(WORK_ROOT / 'runs'),
        name='fire',
        exist_ok=True,

        # Optimizer
        lr0=0.01,
        lrf=0.01,
        warmup_epochs=3,

        # Augmentation
        hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
        degrees=10.0,
        translate=0.1,
        scale=0.5,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.1,

        plots=True,
        verbose=True,
    )

# ---------------------------------------------------------------------------
# Mirror last.pt + best.pt to /kaggle/working/ so they survive into the
# notebook's saved Output even if the next cell never runs.
# ---------------------------------------------------------------------------
for fname in ('last.pt', 'best.pt'):
    src = run_dir / 'weights' / fname
    if src.exists():
        shutil.copy2(src, WORK_ROOT / fname)
        print(f"Mirrored to /kaggle/working/{fname}")

## Step 5 — How good is it?

We run validation on the held-out images and print the headline numbers. **mAP50** is the one to watch — anything above ~0.6 is usable, above ~0.8 is good. Then we run a few sample predictions and display them so you can eyeball the results.

In [ ]:
metrics = model.val(data=str(dst_yaml), imgsz=640, plots=True)
print(f"\nmAP50      : {metrics.box.map50:.3f}")
print(f"mAP50-95   : {metrics.box.map:.3f}")
print(f"Precision  : {metrics.box.mp:.3f}")
print(f"Recall     : {metrics.box.mr:.3f}")

In [ ]:
# Run the trained model on a handful of validation images and show the result
import glob
from IPython.display import Image, display

val_dir = cfg['val']
sample_imgs = sorted(glob.glob(f"{val_dir}/*.jpg"))[:6] or sorted(glob.glob(f"{val_dir}/*.png"))[:6]

if sample_imgs:
    pred_dir = WORK_ROOT / 'runs' / 'predict_sample'
    model.predict(
        source=sample_imgs,
        save=True,
        project=str(pred_dir.parent),
        name=pred_dir.name,
        exist_ok=True,
        conf=0.25,
        verbose=False,
    )
    for p in sorted(pred_dir.glob('*.jpg')):
        display(Image(filename=str(p)))
else:
    print('No sample images found to preview.')

## Step 6 — Save `best.pt` so you can download it

Ultralytics drops the trained weights at `runs/fire/weights/best.pt`. We copy that to the top of `/kaggle/working` so it shows up in the **Output** panel on the right with a one-click download button.

In [ ]:
import shutil

src = WORK_ROOT / 'runs' / 'fire' / 'weights' / 'best.pt'
dst = WORK_ROOT / 'best.pt'

if not src.exists():
    raise FileNotFoundError(f"Training output not found at {src}")

shutil.copy2(src, dst)
size_mb = dst.stat().st_size / (1024 * 1024)
print(f"Saved: {dst}  ({size_mb:.1f} MB)")
print("\nFind it in the Output panel on the right-hand side and click the download icon.")

## Done — next step on your laptop

1. Download `best.pt` from the **Output** panel on the right.
2. Put it in a folder called `models/` at the top of the project.
3. Follow `laptop/README.md` to install and run the detector on a video.